# Master Script — REGRESSÃO (scikit-learn)

**Guia de consulta.** Não foi feito para rodar de ponta a ponta: é um repositório de
sintaxe, pipelines e grades de hiperparâmetros para copiar quando um projeto novo começa.

### Fluxo do documento
| # | Etapa | O que tem aqui |
|---|-------|----------------|
| 0 | Setup | Imports canônicos, `RANDOM_STATE`, config global |
| 1 | Data Prep | Leitura, tipagem, outliers, análise do alvo |
| 2 | Split | `train_test_split`, `KFold`, variantes temporal e por grupo |
| 3 | Pipelines | `ColumnTransformer` + um pipeline por família + alvo transformado |
| 4 | Validação Cruzada | `cross_validate` comparando todas as famílias |
| 5 | Otimização | `RandomizedSearchCV` + **grades de todas as famílias** |
| 6 | Prova Final | RMSE/MAE/R², resíduos, intervalos, importância |
| A | Apêndice | Cheatsheet de famílias, métricas e diagnóstico |

### As três diferenças em relação à classificação
1. **Não existe estratificação** — o `KFold` é embaralhado, não estratificado (mas dá para estratificar por faixa do alvo, ver 2.3).
2. **O alvo também é pré-processado.** Alvo assimétrico (preço, renda, contagem) pede `log1p` ou `TransformedTargetRegressor` — costuma ser o ganho isolado mais alto do projeto.
3. **A análise de resíduos vale mais que a métrica.** Um R² de 0,90 com resíduo em forma de funil é um modelo quebrado.

---
## 0. Setup e imports

In [ ]:
# --- Núcleo ---------------------------------------------------------------
import numpy as np
import pandas as pd

# --- Pré-processamento ----------------------------------------------------
from sklearn.compose import (
    ColumnTransformer, make_column_selector, TransformedTargetRegressor
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler,
    PowerTransformer, QuantileTransformer, FunctionTransformer,
    OneHotEncoder, OrdinalEncoder, TargetEncoder,
    PolynomialFeatures, SplineTransformer,
)
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, RFECV

# --- Split e validação ----------------------------------------------------
from sklearn.model_selection import (
    train_test_split,
    KFold, RepeatedKFold, GroupKFold, TimeSeriesSplit,
    cross_validate, cross_val_predict, learning_curve, validation_curve,
    GridSearchCV, RandomizedSearchCV,
)

# --- Modelos: famílias matemáticas ----------------------------------------
from sklearn.linear_model import (
    LinearRegression,                                  # OLS, sem regularização
    Ridge, Lasso, ElasticNet,                          # regularizados (L2 / L1 / mistura)
    LassoLars, LarsCV, OrthogonalMatchingPursuit,
    BayesianRidge, ARDRegression,                      # bayesianos: dão desvio da predição
    HuberRegressor, RANSACRegressor, TheilSenRegressor, QuantileRegressor,  # robustos
    SGDRegressor, PoissonRegressor, GammaRegressor, TweedieRegressor,       # GLM
    RidgeCV, LassoCV, ElasticNetCV,                    # com CV interna do alpha
)
from sklearn.svm import SVR, LinearSVR, NuSVR
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor, AdaBoostRegressor,
    BaggingRegressor, VotingRegressor, StackingRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor                # baseline obrigatório
from sklearn.kernel_ridge import KernelRidge

# --- Métricas -------------------------------------------------------------
from sklearn.metrics import (
    mean_squared_error, root_mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, median_absolute_error,
    r2_score, explained_variance_score, max_error,
    mean_squared_log_error, mean_pinball_loss, d2_absolute_error_score,
    make_scorer, PredictionErrorDisplay,
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import matplotlib.pyplot as plt
import joblib

# --- Config global --------------------------------------------------------
import os

RANDOM_STATE = 42

# Paralelismo em DOIS níveis, de propósito. Cada worker do joblib abre um pool
# OpenMP do tamanho da máquina: -1 aqui E na validação cruzada gera n_núcleos²
# threads disputando n_núcleos. Com o LightGBM isso não é lentidão, é travamento.
# Paralelize num nível só — o externo rende mais, porque cobre todos os modelos.
N_JOBS       = -1     # cross_validate, RandomizedSearchCV, permutation_importance
N_JOBS_MODEL = 1      # dentro do estimador (KNN, RF, ExtraTrees, XGBoost, LightGBM)

# Modo rápido: usado pelo CI para executar o notebook inteiro em minutos.
# Na leitura normal a variável de ambiente não existe e valem os números cheios.
MODO_RAPIDO    = os.getenv("MASTER_SCRIPT_FAST") == "1"
N_LINHAS       = 300 if MODO_RAPIDO else 1_200   # linhas do DataFrame fictício
N_ITER_BUSCA   = 5   if MODO_RAPIDO else 60      # n_iter do RandomizedSearchCV
N_REPEATS_PERM = 3   if MODO_RAPIDO else 20      # n_repeats da importância por permutação

pd.set_option("display.max_columns", 100)

# A semente vive no Generator (np.random.default_rng, abaixo) e em cada
# random_state=. np.random.seed() é a API legada e não alcança nenhum dos dois.

In [ ]:
# --- Busca sucessiva (experimental) ---------------------------------------
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV, HalvingGridSearchCV

# --- Bibliotecas externas (mesma API do sklearn) --------------------------
# pip install xgboost lightgbm catboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

---
## 1. Carregamento e Limpeza (Data Prep)

Esqueleto do preparo padrão. Dados fictícios, só para dar forma ao resto do documento:
numéricas + categóricas + faltantes + alvo contínuo assimétrico.

In [ ]:
# ==== 1.1 Leitura ==========================================================
# df = pd.read_csv("dados.csv", sep=",", decimal=".", encoding="utf-8")
# df = pd.read_parquet("dados.parquet")
# df = pd.read_sql("SELECT * FROM tabela", con=engine)

# ---- DataFrame fictício (substituir pela leitura real) --------------------
n = N_LINHAS
rng = np.random.default_rng(RANDOM_STATE)

area   = rng.gamma(6, 20, n)                       # m²
quartos = rng.integers(1, 6, n).astype(float)
idade_imovel = rng.integers(0, 60, n).astype(float)
dist_centro  = rng.exponential(8, n)
bairro = rng.choice(["Centro", "Norte", "Sul", "Leste", "Oeste"], n)
tipo   = rng.choice(["apartamento", "casa", "kitnet"], n, p=[.6, .3, .1])
padrao = rng.choice(["popular", "medio", "alto"], n)

# Alvo com relação não-linear + ruído multiplicativo -> distribuição assimétrica.
# O sigma do lognormal controla a assimetria: 0.45 leva o skew a ~1,7, acima do
# limiar de 1 que a etapa 1.2 usa como regra. É o que faz a comparação da 4.3
# (alvo bruto x log1p) ter ganho real para mostrar — o ponto central do notebook.
preco = (1_500 * area
         + 25_000 * quartos
         - 900 * idade_imovel
         - 4_000 * dist_centro
         + rng.normal(0, 40_000, n)) * rng.lognormal(0, .45, n)

df = pd.DataFrame({
    "area": area, "quartos": quartos, "idade_imovel": idade_imovel,
    "dist_centro": dist_centro, "bairro": bairro, "tipo": tipo, "padrao": padrao,
    "id_imovel": np.arange(n),
    "preco": preco.clip(30_000, None),
})

for col, frac in [("area", .06), ("dist_centro", .04), ("padrao", .03)]:
    df.loc[df.sample(frac=frac, random_state=RANDOM_STATE).index, col] = np.nan

df.head()

In [ ]:
# ==== 1.2 Diagnóstico + análise do ALVO ===================================
print(df.shape)
df.info()
display(df.describe().T)

TARGET = "preco"
y_raw = df[TARGET]

print(f"\nassimetria (skew): {y_raw.skew():.3f}   |   curtose: {y_raw.kurt():.3f}")
print("skew > 1  -> forte candidato a log1p / PowerTransformer no alvo")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(y_raw, bins=50, color="#4C72B0"); axes[0].set_title("Alvo bruto")
axes[1].hist(np.log1p(y_raw), bins=50, color="#55A868"); axes[1].set_title("log1p(alvo)")
# orientation= substitui o vert=, depreciado no matplotlib 3.11 e removido no 3.13
axes[2].boxplot(y_raw.dropna(), orientation="horizontal")
axes[2].set_title("Outliers do alvo")
plt.tight_layout(); plt.show()

In [ ]:
# ==== 1.3 Limpeza padrão ==================================================
# a) Identificadores e vazamentos
df = df.drop(columns=["id_imovel"], errors="ignore")

# b) Duplicatas
df = df.drop_duplicates()

# c) Tipagem
CAT_COLS = ["bairro", "tipo", "padrao"]
NUM_COLS = ["area", "quartos", "idade_imovel", "dist_centro"]
df[CAT_COLS] = df[CAT_COLS].astype("category")
df[NUM_COLS] = df[NUM_COLS].apply(pd.to_numeric, errors="coerce")

# d) Linhas sem alvo -> remover (nunca imputar o alvo)
df = df.dropna(subset=[TARGET])

# e) Outliers do ALVO: em regressão eles dominam o RMSE.
#    Três caminhos — escolha um, não os três:
#      1. manter e usar modelo robusto (Huber, RANSAC) ou métrica MAE;
#      2. transformar o alvo (log1p) — reduz o peso da cauda sem descartar dado;
#      3. cortar (só quando for erro de medição comprovado).
q1, q3 = df[TARGET].quantile([.25, .75])
lim_inf, lim_sup = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)   # 3x IQR = corte conservador
print(f"fora do intervalo [{lim_inf:,.0f}, {lim_sup:,.0f}]: "
      f"{((df[TARGET] < lim_inf) | (df[TARGET] > lim_sup)).sum()} linhas")
# df = df[(df[TARGET] >= lim_inf) & (df[TARGET] <= lim_sup)]   # descomentar se for o caso

# f) Correlação: colinearidade forte prejudica interpretação de modelo linear
display(df[NUM_COLS + [TARGET]].corr(numeric_only=True).round(3))

# g) X / y
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)
print(X.shape, y.shape)

---
## 2. Separação dos Dados (Split)

Sem estratificação por padrão. O teste fica lacrado até a etapa 6.

| Situação | Split |
|---|---|
| Tabular comum | `train_test_split(..., shuffle=True)` |
| Alvo muito assimétrico ou base pequena | estratificar por **faixa** do alvo (2.3) |
| Linhas repetidas por entidade | `GroupKFold` / `GroupShuffleSplit` |
| Série temporal | corte por data + `TimeSeriesSplit` (**nunca** embaralhar) |
| Early stopping (XGB/LGBM/MLP) | treino + **validação** + teste |

In [ ]:
# ==== 2.1 Split padrão: 80 / 20 ===========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
)
print(f"treino: {X_train.shape} | teste: {X_test.shape}")
print(f"média do alvo — treino: {y_train.mean():,.0f} | teste: {y_test.mean():,.0f}")

In [ ]:
# ==== 2.2 Split de 3 vias (early stopping / calibração) ===================
X_temp, X_test2, y_temp, y_test2 = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
X_train2, X_vali, y_train2, y_vali = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE   # 60/20/20
)
print(f"treino: {len(X_train2)} | validação: {len(X_vali)} | teste: {len(X_test2)}")

# Uso típico com early stopping (fora do Pipeline, direto no estimador):
# modelo = LGBMRegressor(n_estimators=5000, learning_rate=0.02)
# modelo.fit(prep.fit_transform(X_train2), y_train2,
#            eval_set=[(prep.transform(X_vali), y_vali)],
#            eval_metric="rmse",
#            callbacks=[lightgbm.early_stopping(100, verbose=False)])

In [ ]:
# ==== 2.3 Variantes de split ==============================================

# --- Estratificado por FAIXA do alvo: garante que a cauda apareça nos dois lados
y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.20, stratify=y_bins, random_state=RANDOM_STATE
)

# --- Por grupo ------------------------------------------------------------
# from sklearn.model_selection import GroupShuffleSplit
# gss = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=RANDOM_STATE)
# idx_tr, idx_te = next(gss.split(X, y, groups=df["id_cliente"]))

# --- Temporal -------------------------------------------------------------
# corte = df["data"].quantile(0.8)
# X_train, y_train = X[df["data"] <= corte], y[df["data"] <= corte]
# X_test,  y_test  = X[df["data"] >  corte], y[df["data"] >  corte]

# ==== Estratégia de CV usada da etapa 4 em diante =========================
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Base pequena (mais estável, mais caro):
# CV = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
# Série temporal (só passado prevê futuro):
# CV = TimeSeriesSplit(n_splits=5, gap=0, test_size=None)

---
## 3. Construção dos Pipelines

Mesmas três receitas de `ColumnTransformer` da classificação:

| Receita | Numéricas | Categóricas | Para quem |
|---|---|---|---|
| `"scaled"` | mediana + `StandardScaler` | moda + `OneHotEncoder` | Linear, Ridge/Lasso, SVR, KNN, MLP, GP |
| `"tree"` | mediana, **sem escala** | moda + `OrdinalEncoder` | Árvore, RF, ET, GB, AdaBoost |
| `"native"` | mediana, sem escala | `OrdinalEncoder` categórica nativa | HistGB, LightGBM, XGBoost, CatBoost |

**Específico de regressão:** o alvo também entra na esteira. `TransformedTargetRegressor`
aplica `log1p` no `fit` e `expm1` no `predict` automaticamente — inclusive dentro da CV,
o que evita o vazamento clássico de transformar o alvo antes do split.

In [ ]:
# ==== 3.1 Seleção de colunas ==============================================
NUM_FEATURES = ["area", "quartos", "idade_imovel", "dist_centro"]
CAT_FEATURES = ["bairro", "tipo", "padrao"]

num_selector = make_column_selector(dtype_include=np.number)
cat_selector = make_column_selector(dtype_include=["object", "category"])

In [ ]:
# ==== 3.2 Fábrica de pré-processadores ====================================
def build_preprocessor(kind="scaled", scaler=None, num=NUM_FEATURES, cat=CAT_FEATURES):
    # kind: "scaled" | "tree" | "native"
    if kind == "scaled":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler or StandardScaler()),
            # ("power", PowerTransformer(method="yeo-johnson")),   # variável assimétrica
            # ("spline", SplineTransformer(n_knots=5, degree=3)),  # não-linearidade suave
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                      min_frequency=0.01, sparse_output=False)),
        ])
    elif kind == "tree":
        num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value",
                                       unknown_value=-1, encoded_missing_value=-2)),
        ])
    elif kind == "native":
        num_pipe = Pipeline([("passthrough", "passthrough")])
        cat_pipe = Pipeline([
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value",
                                       unknown_value=-1, encoded_missing_value=np.nan)),
        ])
    else:
        raise ValueError(kind)

    return ColumnTransformer(
        [("num", num_pipe, num), ("cat", cat_pipe, cat)],
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

# --- TargetEncoder: alternativa ao OHE para categórica de alta cardinalidade
# Faz CV interna para não vazar o alvo; sempre dentro do Pipeline.
# cat_pipe_te = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("encoder", TargetEncoder(target_type="continuous", cv=5, random_state=RANDOM_STATE)),
# ])

In [ ]:
# ==== 3.3 Transformação do ALVO ===========================================
# Regra prática: alvo com skew > 1 (preço, renda, tempo, contagem) quase sempre
# melhora com log. O TransformedTargetRegressor devolve a métrica na escala
# ORIGINAL, que é a que interessa para o negócio.

def com_log_no_alvo(pipe):
    return TransformedTargetRegressor(
        regressor=pipe,
        func=np.log1p,           # aplicado em y antes do fit
        inverse_func=np.expm1,   # aplicado na saída do predict
        check_inverse=True,
    )

# Alternativa automática (aprende a melhor transformação nos dados de treino):
# TransformedTargetRegressor(regressor=pipe,
#                            transformer=PowerTransformer(method="yeo-johnson"))
# TransformedTargetRegressor(regressor=pipe,
#                            transformer=QuantileTransformer(output_distribution="normal"))

# ATENÇÃO: aplicar np.log1p(y) manualmente antes do split funciona, mas todas as
# métricas passam a estar em escala log — um RMSE de 0,21 não diz nada ao negócio.

In [ ]:
# ==== 3.4 Um Pipeline por família matemática ==============================
PIPELINES = {

    # ---------------- Baseline -------------------------------------------
    # Prever sempre a mediana. Qualquer modelo precisa vencer isso.
    "Dummy": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", DummyRegressor(strategy="median")),
    ]),

    # ---------------- Lineares (sem e com regularização) ------------------
    "LinearRegression": Pipeline([          # OLS puro: referência interpretável
        ("prep", build_preprocessor("scaled")),
        ("model", LinearRegression()),
    ]),
    "Ridge": Pipeline([                     # L2: encolhe coeficientes, trata colinearidade
        ("prep", build_preprocessor("scaled")),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ]),
    "Lasso": Pipeline([                     # L1: zera coeficientes -> seleção automática
        ("prep", build_preprocessor("scaled")),
        ("model", Lasso(max_iter=10_000, random_state=RANDOM_STATE)),
    ]),
    "ElasticNet": Pipeline([                # L1 + L2: melhor com variáveis correlacionadas
        ("prep", build_preprocessor("scaled")),
        ("model", ElasticNet(max_iter=10_000, random_state=RANDOM_STATE)),
    ]),
    "BayesianRidge": Pipeline([             # regulariza sozinho e devolve desvio-padrão
        ("prep", build_preprocessor("scaled")),
        ("model", BayesianRidge()),
    ]),
    "SGDRegressor": Pipeline([              # escala para milhões de linhas
        ("prep", build_preprocessor("scaled")),
        ("model", SGDRegressor(max_iter=5000, early_stopping=True,
                               random_state=RANDOM_STATE)),
    ]),

    # ---------------- Lineares robustos (outliers no alvo) ----------------
    "Huber": Pipeline([                     # perda quadrática no centro, linear na cauda
        ("prep", build_preprocessor("scaled")),
        ("model", HuberRegressor(max_iter=1000)),
    ]),
    "RANSAC": Pipeline([                    # ignora explicitamente os "outliers"
        ("prep", build_preprocessor("scaled")),
        ("model", RANSACRegressor(random_state=RANDOM_STATE)),
    ]),
    "QuantileRegressor": Pipeline([         # modela a mediana (ou qualquer quantil)
        ("prep", build_preprocessor("scaled")),
        ("model", QuantileRegressor(quantile=0.5, alpha=0.01, solver="highs")),
    ]),

    # ---------------- Linear + não-linearidade explícita ------------------
    "PolyRidge": Pipeline([                 # interações e curvatura mantendo interpretação
        ("prep", build_preprocessor("scaled")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ]),

    # ---------------- GLM (alvo com distribuição específica) --------------
    # Contagens -> Poisson | valores positivos assimétricos -> Gamma | seguros -> Tweedie
    "PoissonGLM": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", PoissonRegressor(max_iter=1000)),
    ]),
    "GammaGLM": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", GammaRegressor(max_iter=1000)),
    ]),

    # ---------------- Margem / kernel -------------------------------------
    "SVR_rbf": Pipeline([                   # forte em base pequena e limpa; O(n²)
        ("prep", build_preprocessor("scaled")),
        ("model", SVR(kernel="rbf")),
    ]),
    "LinearSVR": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", LinearSVR(dual="auto", max_iter=10_000, random_state=RANDOM_STATE)),
    ]),
    "KernelRidge": Pipeline([               # Ridge no espaço do kernel; solução fechada
        ("prep", build_preprocessor("scaled")),
        ("model", KernelRidge(kernel="rbf")),
    ]),

    # ---------------- Baseado em instância --------------------------------
    "KNN": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", KNeighborsRegressor(n_jobs=N_JOBS_MODEL)),
    ]),

    # ---------------- Árvore isolada --------------------------------------
    # Prediz por patamares (nunca extrapola além do intervalo visto no treino).
    "DecisionTree": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
    ]),

    # ---------------- Ensembles: Bagging ----------------------------------
    "RandomForest": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", RandomForestRegressor(n_estimators=500, n_jobs=N_JOBS_MODEL,
                                        random_state=RANDOM_STATE)),
    ]),
    "ExtraTrees": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", ExtraTreesRegressor(n_estimators=500, n_jobs=N_JOBS_MODEL,
                                      random_state=RANDOM_STATE)),
    ]),

    # ---------------- Ensembles: Boosting ---------------------------------
    "GradientBoosting": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),
    "HistGradientBoosting": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", HistGradientBoostingRegressor(
            # Por NOME, não por posição: o prep tem set_output("pandas"), então os
            # nomes chegam intactos. Índices quebrariam em silêncio se as listas
            # NUM_FEATURES / CAT_FEATURES fossem reordenadas.
            categorical_features=CAT_FEATURES,
            early_stopping=True, validation_fraction=0.1,
            random_state=RANDOM_STATE)),
    ]),
    "AdaBoost": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", AdaBoostRegressor(random_state=RANDOM_STATE)),
    ]),

    # ---------------- Rede neural -----------------------------------------
    "MLP": Pipeline([
        ("prep", build_preprocessor("scaled", scaler=MinMaxScaler())),
        ("model", MLPRegressor(max_iter=800, early_stopping=True,
                               random_state=RANDOM_STATE)),
    ]),

    # ---------------- Boosting externo ------------------------------------
    "XGBoost": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", XGBRegressor(n_estimators=600, learning_rate=0.05,
                               tree_method="hist", objective="reg:squarederror",
                               n_jobs=N_JOBS_MODEL, random_state=RANDOM_STATE)),
    ]),
    "LightGBM": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1,
                                n_jobs=N_JOBS_MODEL, random_state=RANDOM_STATE)),
    ]),
    "CatBoost": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6,
                                    thread_count=N_JOBS_MODEL,   # o n_jobs do CatBoost
                                    verbose=0, random_seed=RANDOM_STATE)),
    ]),
}

# Versão com alvo em log — para alvo assimétrico, compare as duas listas.
PIPELINES_LOG = {f"{k}+log": com_log_no_alvo(v) for k, v in PIPELINES.items()}

print(f"{len(PIPELINES)} pipelines declarados (+ {len(PIPELINES_LOG)} com alvo em log)")

In [ ]:
# ==== 3.5 Blocos opcionais ================================================

# --- Seleção de variáveis com CV (escolhe o número de variáveis sozinho) --
pipe_rfecv = Pipeline([
    ("prep", build_preprocessor("tree")),
    ("select", RFECV(estimator=RandomForestRegressor(n_estimators=100,
                                                     random_state=RANDOM_STATE),
                     step=1, cv=3, scoring="neg_root_mean_squared_error")),
    ("model", RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE)),
])

# --- Redução de dimensionalidade / colinearidade extrema -----------------
pipe_pls = Pipeline([                      # PLS: projeta X maximizando covariância com y
    ("prep", build_preprocessor("scaled")),
    ("model", PLSRegression(n_components=5)),
])

# --- Ensembles de segundo nível ------------------------------------------
voting = VotingRegressor(
    estimators=[("ridge", PIPELINES["Ridge"]),
                ("rf", PIPELINES["RandomForest"]),
                ("gb", PIPELINES["HistGradientBoosting"])],
    weights=[1, 2, 3], n_jobs=N_JOBS,
)

stacking = StackingRegressor(
    estimators=[("rf", PIPELINES["RandomForest"]),
                ("gb", PIPELINES["HistGradientBoosting"]),
                ("svr", PIPELINES["SVR_rbf"])],
    final_estimator=RidgeCV(),     # meta-modelo simples evita overfitting no 2º nível
    cv=5, passthrough=False, n_jobs=N_JOBS,
)

# --- Predição com incerteza ----------------------------------------------
# (a) Regressão quantílica: modela o intervalo diretamente
q_baixo = GradientBoostingRegressor(loss="quantile", alpha=0.05, random_state=RANDOM_STATE)
q_alto  = GradientBoostingRegressor(loss="quantile", alpha=0.95, random_state=RANDOM_STATE)
# HistGradientBoostingRegressor(loss="quantile", quantile=0.05)
# LGBMRegressor(objective="quantile", alpha=0.05)

# (b) Bayesiano: BayesianRidge/GaussianProcess devolvem desvio em predict(return_std=True)
gpr = Pipeline([
    ("prep", build_preprocessor("scaled")),
    ("model", GaussianProcessRegressor(
        kernel=ConstantKernel() * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(),
        normalize_y=True, random_state=RANDOM_STATE)),
])   # excelente com poucas centenas de linhas; inviável acima de ~5k (O(n³))

---
## 4. Avaliação das Famílias (Validação Cruzada)

Todas as famílias na mesma CV, **só com dados de treino**.

> No sklearn **todo scorer é "maior é melhor"**. Por isso as métricas de erro vêm com
> prefixo `neg_` (`neg_root_mean_squared_error`). Multiplique por −1 para reportar.

### Qual métrica escolher
| Métrica | Quando usar | Cuidado |
|---|---|---|
| **RMSE** | padrão; erro grande é desproporcionalmente ruim | dominado por outliers |
| **MAE** | erro médio em unidade de negócio; robusto | não pune erro grande |
| **MedAE** | base muito contaminada por outliers | ignora a cauda |
| **MAPE** | erro relativo, comparável entre escalas | explode se `y ≈ 0` |
| **R²** | fração da variância explicada | não diz se o erro é aceitável; pode ser negativo |
| **RMSLE** | alvo de ordens de grandeza diferentes | exige `y > 0` |
| **Pinball** | previsão de quantil / intervalo | específica de regressão quantílica |

Reporte **sempre RMSE + MAE + R² juntos**: sozinha, nenhuma conta a história inteira.

In [ ]:
# ==== 4.1 Métricas avaliadas em cada fold =================================
SCORING = {
    "rmse":  "neg_root_mean_squared_error",
    "mae":   "neg_mean_absolute_error",
    "medae": "neg_median_absolute_error",
    "mape":  "neg_mean_absolute_percentage_error",
    "r2":    "r2",
    # O nome do scorer é "neg_max_error": no sklearn TODO scorer é "maior é
    # melhor", então a versão registrada das métricas de erro é a negativa.
    # "max_error" não existe no registro e derruba o cross_validate inteiro.
    "max_error": "neg_max_error",
}
METRICA_ALVO = "rmse"     # critério de ordenação e de refit na etapa 5

In [ ]:
# ==== 4.2 Torneio entre as famílias =======================================
import warnings, time
from sklearn.exceptions import ConvergenceWarning


def rodar_torneio(dicionario_pipelines, X_tr, y_tr, cv=CV):
    # Filtro com ESCOPO, não global. warnings.filterwarnings("ignore") solto na
    # célula vale para o resto da sessão e some justamente com o
    # ConvergenceWarning — que é o aviso útil: diz que o solver não convergiu
    # (max_iter baixo, escala errada, alpha mal calibrado).
    linhas = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        warnings.simplefilter("ignore", UserWarning)
        warnings.simplefilter("once", ConvergenceWarning)

        for nome, pipe in dicionario_pipelines.items():
            t0 = time.perf_counter()
            try:
                res = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=SCORING,
                                     n_jobs=N_JOBS, return_train_score=True,
                                     error_score="raise")
                linha = {"modelo": nome}
                for m in SCORING:
                    v = res[f"test_{m}"]
                    # scorers "neg_" voltam negativos -> inverter para leitura humana
                    sinal = -1 if SCORING[m].startswith("neg_") else 1
                    linha[f"{m}_mean"] = (sinal * v).mean()
                    linha[f"{m}_std"]  = v.std()
                linha["train_r2"]    = res["train_r2"].mean()
                linha["gap_overfit"] = res["train_r2"].mean() - res["test_r2"].mean()
                linha["fit_time_s"]  = res["fit_time"].mean()
                linhas.append(linha)
                print(f"{nome:<24} RMSE={linha['rmse_mean']:>12,.0f}  "
                      f"R2={linha['r2_mean']:.4f}  {time.perf_counter()-t0:.1f}s")
            except Exception as e:
                print(f"{nome:<24} FALHOU: {type(e).__name__}: {e}")
    return pd.DataFrame(linhas)


ranking = rodar_torneio(PIPELINES, X_train, y_train)
ranking = ranking.sort_values(f"{METRICA_ALVO}_mean").reset_index(drop=True)  # erro: menor é melhor
display(ranking.round(4))

In [ ]:
# ==== 4.3 Vale a pena transformar o alvo? =================================
# Roda o mesmo torneio com log1p no alvo e compara lado a lado.
ranking_log = rodar_torneio(PIPELINES_LOG, X_train, y_train)
ranking_log = ranking_log.sort_values(f"{METRICA_ALVO}_mean").reset_index(drop=True)

comparacao = (pd.concat([ranking.assign(alvo="bruto"),
                         ranking_log.assign(alvo="log1p",
                                            modelo=ranking_log["modelo"].str.replace(r"\+log$", "", regex=True))])
                .pivot(index="modelo", columns="alvo", values=f"{METRICA_ALVO}_mean"))
comparacao["ganho_%"] = (1 - comparacao["log1p"] / comparacao["bruto"]) * 100
display(comparacao.sort_values("ganho_%", ascending=False).round(3))
# Ganho positivo consistente = adote o alvo em log no pipeline vencedor.

In [ ]:
# ==== 4.4 Leitura visual + escolha da vencedora ===========================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top = ranking.head(15).iloc[::-1]
axes[0].barh(top["modelo"], top[f"{METRICA_ALVO}_mean"],
             xerr=top[f"{METRICA_ALVO}_std"], color="#4C72B0", alpha=.85)
axes[0].set_xlabel(METRICA_ALVO.upper()); axes[0].set_title("Erro (menor é melhor)")
axes[0].grid(axis="x", alpha=.3)

top_r2 = ranking.nlargest(15, "r2_mean").iloc[::-1]
axes[1].barh(top_r2["modelo"], top_r2["r2_mean"], color="#55A868", alpha=.85)
axes[1].set_xlabel("R²"); axes[1].set_title("Variância explicada (maior é melhor)")
axes[1].grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

# Empate estatístico -> desempata pelo modelo mais simples/rápido
melhor = ranking.iloc[0]
teto = melhor[f"{METRICA_ALVO}_mean"] + melhor[f"{METRICA_ALVO}_std"]
display(ranking[ranking[f"{METRICA_ALVO}_mean"] <= teto]
        [["modelo", f"{METRICA_ALVO}_mean", f"{METRICA_ALVO}_std", "r2_mean",
          "gap_overfit", "fit_time_s"]].round(4))

MELHOR_FAMILIA = ranking.loc[0, "modelo"]
print(f"\nFamília vencedora: {MELHOR_FAMILIA}")

In [ ]:
# ==== 4.5 Curva de aprendizado (mais dados resolveriam?) ==================
tam, sc_tr, sc_te = learning_curve(
    PIPELINES[MELHOR_FAMILIA], X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=CV, scoring="neg_root_mean_squared_error", n_jobs=N_JOBS,
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tam, -sc_tr.mean(1), "o-", label="treino")
ax.plot(tam, -sc_te.mean(1), "o-", label="validação")
ax.fill_between(tam, -sc_te.mean(1) - sc_te.std(1), -sc_te.mean(1) + sc_te.std(1), alpha=.2)
ax.set_xlabel("nº de amostras de treino"); ax.set_ylabel("RMSE")
ax.set_title("Curva de aprendizado"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Curvas ainda se aproximando -> mais dados ajudam.
# Curvas juntas e com erro alto -> underfitting: precisa de mais capacidade/variáveis.
# Distância grande e estável -> overfitting: mais regularização.

---
## 5. Escolha e Otimização (`RandomizedSearchCV`)

Grades de **todas as famílias** de regressão. Prefixo `model__` porque o estimador é o
passo `"model"` do Pipeline.

> Se o pipeline vencedor estiver embrulhado em `TransformedTargetRegressor`, o prefixo
> ganha mais um nível: `regressor__model__alpha`. Há um utilitário para isso em 5.3.

In [ ]:
# ==== 5.1 Grades de hiperparâmetros — repositório completo ================
from scipy.stats import loguniform, uniform, randint

GRIDS = {

# --------------------------------------------------- LINEARES REGULARIZADOS
"Ridge": {
    "model__alpha":  loguniform(1e-4, 1e4),   # força da penalização L2
    "model__solver": ["auto", "svd", "cholesky", "lsqr", "saga"],
},

"Lasso": {
    "model__alpha":     loguniform(1e-4, 1e2),  # alto demais zera tudo
    "model__selection": ["cyclic", "random"],
    "model__max_iter":  [5000, 10_000, 20_000],
},

"ElasticNet": {
    "model__alpha":    loguniform(1e-4, 1e2),
    "model__l1_ratio": uniform(0, 1),           # 0 = Ridge puro | 1 = Lasso puro
    "model__selection": ["cyclic", "random"],
},

"BayesianRidge": {
    "model__alpha_1":  loguniform(1e-8, 1e-3),
    "model__alpha_2":  loguniform(1e-8, 1e-3),
    "model__lambda_1": loguniform(1e-8, 1e-3),
    "model__lambda_2": loguniform(1e-8, 1e-3),
},

"SGDRegressor": {
    "model__loss":          ["squared_error", "huber", "epsilon_insensitive"],
    "model__penalty":       ["l1", "l2", "elasticnet"],
    "model__alpha":         loguniform(1e-7, 1e-1),
    "model__l1_ratio":      uniform(0, 1),
    "model__learning_rate": ["invscaling", "adaptive", "constant"],
    "model__eta0":          loguniform(1e-4, 1e-1),
},

# -------------------------------------------------------- LINEARES ROBUSTOS
"Huber": {
    "model__epsilon": uniform(1.0, 2.5),   # menor = mais robusto (1.35 é o padrão)
    "model__alpha":   loguniform(1e-6, 1e-1),
},

"RANSAC": {
    "model__min_samples":     uniform(0.1, 0.8),
    "model__residual_threshold": [None, 1.0, 5.0],
    "model__max_trials":      randint(50, 500),
},

"QuantileRegressor": {
    "model__quantile": [0.1, 0.25, 0.5, 0.75, 0.9],
    "model__alpha":    loguniform(1e-4, 1e1),
},

# ------------------------------------------------------------ POLINOMIAL/GLM
"PolyRidge": {
    "poly__degree":           [2, 3],
    "poly__interaction_only": [True, False],
    "model__alpha":           loguniform(1e-3, 1e4),
},

"PoissonGLM": {
    "model__alpha": loguniform(1e-4, 1e2),
},
"GammaGLM": {
    "model__alpha": loguniform(1e-4, 1e2),
},
# Tweedie: "model__power": uniform(1, 1)  -> 1=Poisson, 2=Gamma, (1,2)=composto

# ---------------------------------------------------------------- KERNEL/SVM
"SVR_rbf": {
    "model__C":       loguniform(1e-1, 1e4),  # penalidade por erro fora do tubo
    "model__gamma":   loguniform(1e-5, 1e1),
    "model__epsilon": loguniform(1e-3, 1e1),  # largura do tubo insensível
    "model__kernel":  ["rbf", "poly", "sigmoid"],
    "model__degree":  randint(2, 5),
},

"LinearSVR": {
    "model__C":       loguniform(1e-3, 1e3),
    "model__epsilon": loguniform(1e-3, 1e1),
    "model__loss":    ["epsilon_insensitive", "squared_epsilon_insensitive"],
},

"KernelRidge": {
    "model__alpha":  loguniform(1e-4, 1e2),
    "model__gamma":  loguniform(1e-5, 1e1),
    "model__kernel": ["rbf", "laplacian", "polynomial"],
    "model__degree": randint(2, 5),
},

# ---------------------------------------------------------------- DISTÂNCIA
"KNN": {
    "model__n_neighbors": randint(1, 50),
    "model__weights":     ["uniform", "distance"],
    "model__metric":      ["euclidean", "manhattan", "minkowski", "chebyshev"],
    "model__p":           [1, 2],
    "model__leaf_size":   randint(10, 60),
},

# ------------------------------------------------------------------- ÁRVORE
"DecisionTree": {
    "model__criterion":         ["squared_error", "friedman_mse",
                                 "absolute_error", "poisson"],
    "model__max_depth":         [None, 3, 5, 8, 12, 20],
    "model__min_samples_split": randint(2, 40),
    "model__min_samples_leaf":  randint(1, 30),
    "model__max_features":      [None, "sqrt", "log2", 0.5],
    "model__ccp_alpha":         loguniform(1e-5, 1e-1),
},

# -------------------------------------------------------- ENSEMBLE: BAGGING
"RandomForest": {
    "model__n_estimators":      randint(200, 1200),
    "model__max_depth":         [None, 5, 10, 20, 40],
    "model__min_samples_split": randint(2, 20),
    "model__min_samples_leaf":  randint(1, 15),
    "model__max_features":      [1.0, "sqrt", "log2", 0.3, 0.5],  # 1.0 é o padrão em regressão
    "model__bootstrap":         [True, False],
    "model__max_samples":       [None, 0.6, 0.8],
},

"ExtraTrees": {
    "model__n_estimators":      randint(200, 1200),
    "model__max_depth":         [None, 10, 20, 40],
    "model__min_samples_split": randint(2, 20),
    "model__min_samples_leaf":  randint(1, 15),
    "model__max_features":      [1.0, "sqrt", "log2", 0.5],
},

# ------------------------------------------------------- ENSEMBLE: BOOSTING
"GradientBoosting": {
    "model__loss":              ["squared_error", "absolute_error", "huber", "quantile"],
    "model__n_estimators":      randint(100, 900),
    "model__learning_rate":     loguniform(1e-3, 0.3),
    "model__max_depth":         randint(2, 8),
    "model__min_samples_leaf":  randint(1, 30),
    "model__subsample":         uniform(0.6, 0.4),
    "model__max_features":      ["sqrt", "log2", None],
    "model__alpha":             uniform(0.1, 0.8),   # só para loss huber/quantile
},

"HistGradientBoosting": {
    "model__loss":              ["squared_error", "absolute_error", "poisson"],
    "model__learning_rate":     loguniform(1e-3, 0.3),
    "model__max_iter":          randint(100, 900),
    "model__max_leaf_nodes":    randint(10, 128),
    "model__max_depth":         [None, 4, 8, 16],
    "model__min_samples_leaf":  randint(5, 60),
    "model__l2_regularization": loguniform(1e-6, 1e1),
    "model__max_bins":          [64, 128, 255],
},

"AdaBoost": {
    "model__n_estimators":  randint(50, 600),
    "model__learning_rate": loguniform(1e-3, 2.0),
    "model__loss":          ["linear", "square", "exponential"],
    "model__estimator":     [DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
                             for d in (2, 3, 5, 8)],
},

# ------------------------------------------------------------- REDE NEURAL
"MLP": {
    "model__hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              loguniform(1e-6, 1e-1),
    "model__learning_rate_init": loguniform(1e-4, 1e-1),
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, 256],
    "model__solver":             ["adam", "lbfgs"],
},

# --------------------------------------------------------- BOOSTING EXTERNO
"XGBoost": {
    "model__n_estimators":     randint(200, 1500),
    "model__learning_rate":    loguniform(1e-3, 0.3),
    "model__max_depth":        randint(2, 12),
    "model__min_child_weight": randint(1, 20),
    "model__subsample":        uniform(0.5, 0.5),
    "model__colsample_bytree": uniform(0.4, 0.6),
    "model__gamma":            loguniform(1e-4, 5),
    "model__reg_alpha":        loguniform(1e-4, 10),
    "model__reg_lambda":       loguniform(1e-3, 50),
    # alvo assimétrico positivo: "model__objective": ["reg:squarederror",
    #                                                 "reg:absoluteerror",
    #                                                 "reg:gamma", "reg:tweedie"]
},

"LightGBM": {
    "model__n_estimators":      randint(200, 1500),
    "model__learning_rate":     loguniform(1e-3, 0.3),
    "model__num_leaves":        randint(15, 255),
    "model__max_depth":         [-1, 4, 8, 16],
    "model__min_child_samples": randint(5, 100),
    "model__subsample":         uniform(0.5, 0.5),
    "model__subsample_freq":    [0, 1, 5],
    "model__colsample_bytree":  uniform(0.4, 0.6),
    "model__reg_alpha":         loguniform(1e-4, 10),
    "model__reg_lambda":        loguniform(1e-4, 10),
    # "model__objective": ["regression", "regression_l1", "huber", "quantile", "tweedie"]
},

"CatBoost": {
    "model__iterations":          randint(200, 1500),
    "model__learning_rate":       loguniform(1e-3, 0.3),
    "model__depth":               randint(4, 10),
    "model__l2_leaf_reg":         loguniform(1e-1, 30),
    "model__border_count":        [32, 64, 128, 254],
    "model__random_strength":     loguniform(1e-2, 10),
    "model__bagging_temperature": uniform(0, 1),
},
}

In [ ]:
# ==== 5.2 Grades dos blocos de pré-processamento ==========================
GRID_PREP = {
    "prep__num__imputer__strategy": ["mean", "median", "most_frequent"],
    "prep__num__scaler":            [StandardScaler(), RobustScaler(), MinMaxScaler(),
                                     PowerTransformer(method="yeo-johnson"),
                                     QuantileTransformer(output_distribution="normal")],
    "prep__cat__encoder__min_frequency": [0.005, 0.01, 0.05],
    # "select__k": randint(5, 25),
    # "poly__degree": [1, 2, 3],
}

In [ ]:
# ==== 5.3 A busca, no pipeline vencedor, só com dados de TREINO ===========
USAR_LOG_NO_ALVO = False     # ligar se a comparação da etapa 4.3 mostrou ganho

# Nem toda família tem o que otimizar. Se um baseline (ou o OLS, que não tem
# hiperparâmetro nenhum) vence o torneio, o problema está nos dados ou nas
# variáveis — não nos hiperparâmetros.
GRIDS_VAZIAS = {"Dummy", "LinearRegression"}

pipe_vencedor = PIPELINES[MELHOR_FAMILIA]
if MELHOR_FAMILIA in GRIDS_VAZIAS or MELHOR_FAMILIA not in GRIDS:
    print(f"{MELHOR_FAMILIA} não tem grade de busca — a etapa 5 vira um refit simples.")
    grid_vencedor = {}
else:
    grid_vencedor = GRIDS[MELHOR_FAMILIA]

if USAR_LOG_NO_ALVO:
    pipe_vencedor = com_log_no_alvo(pipe_vencedor)
    # TransformedTargetRegressor acrescenta o nível "regressor__" aos parâmetros
    grid_vencedor = {f"regressor__{k}": v for k, v in grid_vencedor.items()}

search = RandomizedSearchCV(
    estimator=pipe_vencedor,
    param_distributions=grid_vencedor,
    n_iter=N_ITER_BUSCA if grid_vencedor else 1,
    scoring=SCORING,
    refit=METRICA_ALVO,
    cv=CV,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=True,
    error_score=np.nan,
)

search.fit(X_train, y_train)     # <- o teste continua lacrado

print(f"Melhor {METRICA_ALVO} em CV: {abs(search.best_score_):,.2f}")
print("Melhores parâmetros:")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

best_model = search.best_estimator_

In [ ]:
# ==== 5.4 Inspeção da busca ===============================================
cv_df = (pd.DataFrame(search.cv_results_)
           .sort_values(f"rank_test_{METRICA_ALVO}")
           .loc[:, lambda d: [c for c in d.columns
                              if c.startswith("param_")
                              or c in (f"mean_test_{METRICA_ALVO}",
                                       f"std_test_{METRICA_ALVO}",
                                       f"mean_train_{METRICA_ALVO}",
                                       "mean_fit_time")]])
display(cv_df.head(10).round(4))

# Sensibilidade: qual hiperparâmetro realmente move a métrica?
for col in [c for c in cv_df.columns if c.startswith("param_")][:4]:
    sub = cv_df[[col, f"mean_test_{METRICA_ALVO}"]].dropna().copy()
    # Alguns parâmetros são OBJETOS, não escalares: o estimator do AdaBoost, o
    # scaler do GRID_PREP. Agrupar por eles direto levanta TypeError, porque
    # estimadores não têm ordem definida. Em texto agrupam igual e saem legíveis.
    sub[col] = sub[col].astype(str)
    if sub[col].nunique() > 1:
        print(f"\n{col}")
        print(sub.groupby(col)[f"mean_test_{METRICA_ALVO}"].mean().sort_values(ascending=False).head())

In [ ]:
# ==== 5.5 Variantes de busca ==============================================

# --- Refino local ---------------------------------------------------------
# fino = GridSearchCV(pipe_vencedor,
#                     {"model__max_depth": [6, 8, 10], "model__learning_rate": [.03, .05, .08]},
#                     scoring="neg_root_mean_squared_error", cv=CV, n_jobs=N_JOBS)
# fino.fit(X_train, y_train)

# --- Halving (bem mais rápido) -------------------------------------------
# halving = HalvingRandomSearchCV(pipe_vencedor, grid_vencedor, n_candidates=200,
#                                 factor=3, resource="n_samples",
#                                 scoring="neg_root_mean_squared_error",
#                                 cv=CV, n_jobs=N_JOBS, random_state=RANDOM_STATE)

# --- Atalho: modelos lineares com CV interna do alpha (muito mais barato) --
# RidgeCV(alphas=np.logspace(-4, 4, 100), cv=CV)
# LassoCV(n_alphas=200, cv=CV, n_jobs=N_JOBS, random_state=RANDOM_STATE)
# ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], cv=CV, n_jobs=N_JOBS)

# --- Busca bayesiana (Optuna) --------------------------------------------
# import optuna
# def objective(trial):
#     p = {"model__num_leaves": trial.suggest_int("model__num_leaves", 15, 255),
#          "model__learning_rate": trial.suggest_float("model__learning_rate", 1e-3, .3, log=True)}
#     s = cross_validate(pipe_vencedor.set_params(**p), X_train, y_train,
#                        cv=CV, scoring="neg_root_mean_squared_error")
#     return -s["test_score"].mean()
# optuna.create_study(direction="minimize").optimize(objective, n_trials=100)

---
## 6. A Prova Final (Inferência no Teste)

Teste aberto **uma única vez**. Em regressão, o gráfico de resíduos vale mais que
qualquer número isolado: um R² alto com resíduo estruturado é um modelo com viés.

In [ ]:
# ==== 6.1 Predição e métricas =============================================
y_pred = best_model.predict(X_test)

# Nº de features DEPOIS do encoding. O R² ajustado penaliza pelo número de
# preditores que o modelo realmente usa, não pelo número de colunas do DataFrame
# bruto — o OneHotEncoder pode multiplicar uma coluna por dez.
_pipe = best_model.regressor_ if hasattr(best_model, "regressor_") else best_model
N_FEATURES = _pipe[:-1].transform(X_test).shape[1]

metricas_finais = {
    "RMSE":  root_mean_squared_error(y_test, y_pred),
    "MAE":   mean_absolute_error(y_test, y_pred),
    "MedAE": median_absolute_error(y_test, y_pred),
    "MAPE":  mean_absolute_percentage_error(y_test, y_pred),
    "R2":    r2_score(y_test, y_pred),
    "R2_ajustado": 1 - (1 - r2_score(y_test, y_pred)) * (len(y_test) - 1)
                       / (len(y_test) - N_FEATURES - 1),
    "Explained_Var": explained_variance_score(y_test, y_pred),
    "Max_Error": max_error(y_test, y_pred),
}
display(pd.Series(metricas_finais).round(4).to_frame("teste"))

# Referências que dão sentido aos números
print(f"média do alvo no teste:   {y_test.mean():,.2f}")
print(f"desvio-padrão do alvo:    {y_test.std():,.2f}")
print(f"RMSE / desvio-padrão:     {metricas_finais['RMSE']/y_test.std():.3f}  (< 1 = melhor que a média)")
print(f"MAE como % da média:      {metricas_finais['MAE']/y_test.mean()*100:.2f}%")

# Baseline: prever sempre a média do treino
rmse_base = root_mean_squared_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"\nRMSE do baseline (média): {rmse_base:,.2f}")
print(f"Redução do erro:          {(1 - metricas_finais['RMSE']/rmse_base)*100:.1f}%")

print(f"\nCV (treino): {abs(search.best_score_):,.2f}  |  Teste: {metricas_finais['RMSE']:,.2f}")

In [ ]:
# ==== 6.2 A "matriz de confusão" da regressão: real x previsto + resíduos ==
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

PredictionErrorDisplay.from_predictions(
    y_test, y_pred, kind="actual_vs_predicted", ax=axes[0], scatter_kwargs={"alpha": .5})
axes[0].set_title("Real x Previsto (ideal = diagonal)")

PredictionErrorDisplay.from_predictions(
    y_test, y_pred, kind="residual_vs_predicted", ax=axes[1], scatter_kwargs={"alpha": .5})
axes[1].set_title("Resíduo x Previsto (ideal = nuvem sem forma)")

plt.tight_layout(); plt.show()

# COMO LER O GRÁFICO DE RESÍDUOS
#   Nuvem horizontal sem padrão .... modelo bem especificado
#   Forma de funil (cone) .......... heterocedasticidade -> transformar o alvo (log)
#   Curva em U ..................... falta não-linearidade -> poly/spline ou modelo de árvore
#   Deslocamento sistemático ....... viés: subestima alto / superestima baixo
#   Pontos muito distantes ......... outliers -> investigar ou usar modelo robusto

In [ ]:
# ==== 6.3 Distribuição do erro ============================================
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(residuos, bins=50, color="#4C72B0")
axes[0].axvline(0, c="k", ls="--")
axes[0].set_title(f"Resíduos (média={residuos.mean():,.0f})")

from scipy import stats
stats.probplot(residuos, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot (resíduo normal fica na reta)")

erro_pct = (residuos / y_test).abs() * 100
axes[2].hist(erro_pct.clip(0, 100), bins=50, color="#C44E52")
axes[2].set_title("Erro percentual absoluto")
axes[2].set_xlabel("%")

plt.tight_layout(); plt.show()

# Erro por faixa do alvo: onde o modelo é ruim?
faixas = pd.qcut(y_test, 5, duplicates="drop")
por_faixa = pd.DataFrame({"real": y_test, "pred": y_pred, "faixa": faixas})
resumo = por_faixa.groupby("faixa", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "RMSE": root_mean_squared_error(g["real"], g["pred"]),
        "MAE": mean_absolute_error(g["real"], g["pred"]),
        "vies_medio": (g["pred"] - g["real"]).mean(),
    }), include_groups=False)
display(resumo.round(2))
# Viés médio positivo numa faixa = o modelo superestima aquele intervalo.

In [ ]:
# ==== 6.4 Intervalo de predição ===========================================
# Uma previsão pontual sem incerteza tem uso limitado em decisão.

# (a) Regressão quantílica: dois modelos, um por extremo do intervalo
prep = build_preprocessor("tree")
Xtr_p, Xte_p = prep.fit_transform(X_train, y_train), prep.transform(X_test)
lo = GradientBoostingRegressor(loss="quantile", alpha=0.05,
                               random_state=RANDOM_STATE).fit(Xtr_p, y_train)
hi = GradientBoostingRegressor(loss="quantile", alpha=0.95,
                               random_state=RANDOM_STATE).fit(Xtr_p, y_train)
# q05 / q95, e não lim_inf / lim_sup: esses nomes já são as bordas do corte por
# IQR definidas na etapa 1.3, e sobrescrevê-los apagaria aquele diagnóstico.
q05, q95 = lo.predict(Xte_p), hi.predict(Xte_p)
cobertura = ((y_test >= q05) & (y_test <= q95)).mean()
print(f"cobertura do intervalo de 90%: {cobertura:.1%}   (alvo: ~90%)")
print(f"largura mediana do intervalo:  {np.median(q95 - q05):,.0f}")

# (b) Dispersão entre as árvores de uma floresta
# rf = PIPELINES['RandomForest'].fit(X_train, y_train)
# preds = np.stack([t.predict(rf[:-1].transform(X_test)) for t in rf["model"].estimators_])
# std_pred = preds.std(axis=0)

# (c) Bayesiano: desvio analítico
# BayesianRidge().fit(Xtr_p, y_train).predict(Xte_p, return_std=True)

# Visualização (ordenado pelo valor previsto)
ordem = np.argsort(y_pred)[:200]
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(range(len(ordem)), q05[ordem], q95[ordem],
                alpha=.3, color="#4C72B0", label="intervalo 90%")
ax.plot(y_pred[ordem], lw=1.5, color="#4C72B0", label="previsto")
ax.scatter(range(len(ordem)), y_test.values[ordem], s=8, color="#C44E52", label="real")
ax.legend(); ax.set_title("Predição com intervalo (200 amostras)")
plt.tight_layout(); plt.show()

In [ ]:
# ==== 6.5 Importância das variáveis e efeito parcial ======================
imp = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_root_mean_squared_error", n_repeats=N_REPEATS_PERM,
    random_state=RANDOM_STATE, n_jobs=N_JOBS)
imp_df = (pd.DataFrame({"variavel": X_test.columns,
                        "importancia": imp.importances_mean,
                        "std": imp.importances_std})
            .sort_values("importancia", ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
top = imp_df.head(15).iloc[::-1]
ax.barh(top["variavel"], top["importancia"], xerr=top["std"], color="#55A868")
ax.set_title("Importância por permutação (aumento do RMSE ao embaralhar)")
ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

# --- Dependência parcial: COMO cada variável afeta a previsão -------------
PartialDependenceDisplay.from_estimator(
    best_model, X_test, features=list(imp_df["variavel"].head(3)),
    kind="both",     # "average" = PDP | "individual" = ICE | "both"
    n_jobs=N_JOBS, grid_resolution=30)
plt.tight_layout(); plt.show()

# --- Coeficientes (modelo linear) ----------------------------------------
# nomes = best_model.named_steps["prep"].get_feature_names_out()
# coefs = pd.Series(best_model.named_steps["model"].coef_, index=nomes)
# display(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(20).round(4))
# Com StandardScaler os coeficientes são comparáveis entre si.

In [ ]:
# ==== 6.6 Piores erros ====================================================
erros = X_test.copy()
erros["real"] = y_test.values
erros["previsto"] = y_pred
erros["residuo"] = residuos.values
erros["erro_pct"] = (erros["residuo"] / erros["real"] * 100).round(1)

print("Maiores subestimativas (modelo previu muito abaixo do real):")
display(erros.nlargest(5, "residuo"))
print("Maiores superestimativas:")
display(erros.nsmallest(5, "residuo"))

In [ ]:
# ==== 6.7 Modelo final e persistência =====================================
# Re-treino nos dados COMPLETOS: o teste já cumpriu seu papel de estimativa.
from sklearn.base import clone
modelo_producao = clone(search.best_estimator_)   # copia a configuração, descarta o que já foi ajustado
modelo_producao.fit(X, y)

artefato = {
    "modelo": modelo_producao,
    "features": {"num": NUM_FEATURES, "cat": CAT_FEATURES},
    "alvo": TARGET,
    "alvo_em_log": USAR_LOG_NO_ALVO,
    "metricas_teste": metricas_finais,
    "params": search.best_params_,
    "faixa_treino_alvo": (float(y.min()), float(y.max())),  # fora dela, extrapolação
    "sklearn_version": __import__("sklearn").__version__,
    "treinado_em": pd.Timestamp.now().isoformat(),
}
joblib.dump(artefato, "modelo_regressao.joblib", compress=3)

# --- Carregar e usar ------------------------------------------------------
# art = joblib.load("modelo_regressao.joblib")
# y_novo = art["modelo"].predict(X_novo)
#
# LEMBRETE: modelos de árvore NÃO extrapolam. Entrada fora de
# art["faixa_treino_alvo"] devolve o valor da folha mais próxima, não uma
# extensão da tendência. Para extrapolar, use modelo linear.

---
## Apêndice — Cheatsheet

### Qual família tentar primeiro
| Situação | Comece por |
|---|---|
| Tabular, qualquer tamanho | **HistGradientBoosting / LightGBM / CatBoost** |
| Precisa explicar e extrapolar | `Ridge`, `ElasticNet`, `LinearRegression` |
| Poucas linhas (< 500) | `Ridge`, `BayesianRidge`, `SVR`, `GaussianProcess` |
| Muitas colunas, poucas linhas (p >> n) | `Lasso`, `ElasticNet`, `PLSRegression` |
| Muitas linhas (> 1M) | `SGDRegressor`, `LightGBM`, `HistGradientBoosting` |
| Outliers pesados no alvo | `HuberRegressor`, `RANSAC`, `QuantileRegressor`, `loss="absolute_error"` |
| Alvo = contagem (0,1,2,…) | `PoissonRegressor`, `HistGB(loss="poisson")` |
| Alvo positivo assimétrico (preço, sinistro) | `GammaRegressor`, `TweedieRegressor`, ou `log1p` no alvo |
| Precisa de intervalo de predição | Regressão quantílica, `BayesianRidge`, `GaussianProcess` |
| Relação suave e não-linear | `SplineTransformer` + `Ridge`, `SVR(rbf)`, `GaussianProcess` |

### Sintomas e correções
| Sintoma | Causa provável | Correção |
|---|---|---|
| R² negativo | pior que prever a média | modelo/pré-processo errado; verifique vazamento invertido |
| Resíduo em funil | heterocedasticidade | `log1p` no alvo ou `TransformedTargetRegressor` |
| Resíduo em U | falta não-linearidade | `PolynomialFeatures`, `SplineTransformer`, árvore |
| RMSE >> MAE | poucos erros enormes | outliers: modelo robusto ou corte investigado |
| CV ótima, teste ruim | vazamento ou busca superajustada | pré-processo no Pipeline; reduza `n_iter` |
| Métrica boa, negócio insatisfeito | métrica errada | troque para MAE/MAPE ou avalie por faixa do alvo |
| Previsão travada num teto | árvore não extrapola | modelo linear ou híbrido |

### Multi-alvo (prever várias colunas de uma vez)
```python
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
MultiOutputRegressor(GradientBoostingRegressor())   # um modelo por alvo, independentes
RegressorChain(Ridge())                             # cada alvo usa a previsão do anterior
# RandomForest, ExtraTrees, KNN e Ridge já aceitam y com 2 dimensões nativamente.
```

### Séries temporais — o que muda
```python
CV = TimeSeriesSplit(n_splits=5, gap=0)   # nunca embaralhar
# Variáveis obrigatórias: defasagens (lags), médias móveis, sazonalidade (seno/cosseno),
# calendário. Valide sempre com corte temporal, nunca com KFold aleatório.
```